# ⚙️ AIOS + Genuine Kali Linux Toolset in Google Colab

This notebook equips Google Colab with the **official Kali Linux security toolset** (`kali-tools-top10` / `kali-linux-headless`), compiles the **AIOS Security Operating Layer**, and launches a graphical desktop with local AI.

### Instructions:
1. **Runtime** ➔ **Change runtime type** ➔ **T4 GPU** ➔ **Save**.
2. Run cells 1 through 6 in order.
3. Step 4 gives you the graphical desktop URL.

### Step 1: Clone AIOS Repository

In [ ]:
!git clone https://github.com/Habib112233445566/AIOS.git /content/AIOS
%cd /content/AIOS
!ls -la

### Step 2: Add Official Kali Linux Repositories & Install Genuine Kali Toolset

This pulls directly from `http.kali.org/kali` with Kali's cryptographic archive keyring.

In [ ]:
# 1. Add official Kali Linux GPG signing key
!wget -q -O - https://archive.kali.org/archive-key.asc | gpg --dearmor -o /etc/apt/trusted.gpg.d/kali-archive-keyring.gpg
!echo "deb http://http.kali.org/kali kali-rolling main contrib non-free non-free-firmware" > /etc/apt/sources.list.d/kali.list

# 2. Configure Apt Pinning to prioritize Kali security tools safely
apt_pin = """Package: *
Pin: release o=Kali
Pin-Priority: 100

Package: lib* python3* linux-*
Pin: release o=Kali
Pin-Priority: -1
"""
with open('/etc/apt/preferences.d/kali.pref', 'w') as f:
    f.write(apt_pin)

# 3. Update package index and install Kali security tools & Rust
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends \
    nmap sqlmap wireshark-common tshark nikto john aircrack-ng hydra responder curl wget build-essential git python3-pip

# 4. Install Rust compiler for AIOS
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] += ":/root/.cargo/bin"
!rustc --version

### Step 3: Build AIOS Linux Operating Layer

In [ ]:
%cd /content/AIOS/code/aiosh-rust
!cargo build --release -p aiosh-cli -p aiosh-mcp
!ls -lh target/release/aiosh

### Step 4: Launch the Desktop & noVNC Web Link

In [ ]:
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xfce4 xfce4-goodies tigervnc-standalone-server novnc websockify

!mkdir -p ~/.vnc
!echo "aios1234" | vncpasswd -f > ~/.vnc/passwd
!chmod 600 ~/.vnc/passwd

vnc_startup = """#!/bin/bash
xrdb $HOME/.Xresources
startxfce4 &
"""
with open('/root/.vnc/xstartup', 'w') as f:
    f.write(vnc_startup)
!chmod +x /root/.vnc/xstartup

!vncserver -kill :1 2>/dev/null || true
!vncserver :1 -geometry 1280x800 -depth 24

import subprocess, time, re
subprocess.Popen(["websockify", "--web", "/usr/share/novnc/", "6080", "localhost:5901"])
time.sleep(2)

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

print("Starting Cloudflare Tunnel to generate web URL...")
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:6080/vnc.html"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

tunnel_url = None
for _ in range(30):
    line = tunnel.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0) + "/vnc.html?autoconnect=true&resize=scale"
        break
    time.sleep(0.5)

if tunnel_url:
    print("\n" + "="*70)
    print("\U0001f5a5️ CLICK THIS LINK TO ACCESS YOUR DESKTOP WITH KALI TOOLS:")
    print(f"   {tunnel_url}")
    print("   VNC Password (if asked): aios1234")
    print("="*70 + "\n")
else:
    print("Tunnel started. Check Cloudflare output.")

### Step 5: Start Local AI Engine (Ollama)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

!ollama pull llama3.2:1b
!ollama list

### Step 6: Test AIOS Operating Layer with Kali Tools

In [ ]:
%cd /content/AIOS/code/aiosh-rust
!./target/release/aiosh --help

# Give the AI a real security audit task using the installed Kali tools
!./target/release/aiosh agent --goal "Check open network services on localhost using nmap and report active listening ports."